In [53]:
# This model uses RegexpParser for analyzing user queries to check whether its a correct grammer or not
# If correct grammer then it uses the query as it is otherwise it classifies it into 'other' category
# If the query matches the grammer then it identifies category of the query

import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Download necessary NLTK datasets if you haven't already
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

#Grammar validation
def validate_grammer(text):
    """
    Cleans text by isolating core Noun-Verb relationships using RegexpParser.
    Funnels out fluff words like adverbs, determiners, etc.
    """
    # Tokenize and POS tag the text
    tokens = nltk.word_tokenize(text)
    pos_tags = nltk.pos_tag(tokens)

    # Define a grammar: Looking for optional Nouns followed by a Verb
    # Rules: <NN.*> matches any noun (NN, NNS, NNP), <VB.*> matches any verb
    grammar = r"""
        CoreAction: {<NN.*>*<VB.*>+}
    """
    parser = nltk.RegexpParser(grammar)
    tree = parser.parse(pos_tags)

    # Extract only the words that fell inside our 'CoreAction' chunks
    clean_words = []
    for subtree in tree.subtrees(filter=lambda t: t.label() == 'CoreAction'):
        for word, tag in subtree.leaves():
            clean_words.append(word.lower())

    # If the sentence matches the grammer use text as it is else return 'unknown'
    if len(clean_words):
        return text.lower()
    else:
        return 'unknown'


# --- 2. PREPARE THE TRAINING DATA ---
# Imagine a dataset classifying technical support vs billing issues
corpus = [
    "My screen suddenly went completely blank and the computer crashed",
    "I would like to request a full refund for my subscription",
    "The system keeps throwing a strange 404 error during login",
    "Please cancel my account charges immediately",
    "The application is lagging terribly and won't open",
    "Please send a copy of last month billing statement",
    "Check due or balance on my account",
]
labels = ["tech", "billing", "tech", "billing", "tech", 'billing', 'billing']

cleaned_corpus = [validate_grammer(text) for text in corpus]
print(cleaned_corpus)

# --- 4. TRAIN THE MACHINE LEARNING MODEL ---
# We build a pipeline that converts text to TF-IDF vectors and runs Logistic Regression
ml_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', LogisticRegression())
])

# Train on our clean features
ml_pipeline.fit(cleaned_corpus, labels)


# --- 5. PREDICT ON NEW UNSEEN TEXT ---
new_complaints = [
    "The software crashed instantly",
    "Can you check my billing statement?",
    "Can you renew my account",
    "Please check balance of my account",
    "Can you please send me last month payment statement ?",
    "Are you sure?"
]

# Clean the new incoming data using the exact same parser logic
cleaned_new_complaints = [validate_grammer(text) for text in new_complaints]
print(cleaned_new_complaints)

# Predict
predictions = ml_pipeline.predict(cleaned_new_complaints)

print("--- Final Predictions ---")
for doc, cleaned_doc, pred in zip(new_complaints, cleaned_new_complaints, predictions):
    if cleaned_doc == 'unknown':
      pred='other'
    print(f"Document: '{doc}' -> Predicted Category: {pred.upper()}")

['my screen suddenly went completely blank and the computer crashed', 'i would like to request a full refund for my subscription', 'the system keeps throwing a strange 404 error during login', 'please cancel my account charges immediately', "the application is lagging terribly and won't open", 'please send a copy of last month billing statement', 'check due or balance on my account']
['the software crashed instantly', 'can you check my billing statement?', 'can you renew my account', 'please check balance of my account', 'can you please send me last month payment statement ?', 'unknown']
--- Final Predictions ---
Document: 'The software crashed instantly' -> Predicted Category: TECH
Document: 'Can you check my billing statement?' -> Predicted Category: BILLING
Document: 'Can you renew my account' -> Predicted Category: BILLING
Document: 'Please check balance of my account' -> Predicted Category: BILLING
Document: 'Can you please send me last month payment statement ?' -> Predicted Cate

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
